In [5]:
import os
import sys

txpipe_dir = os.path.join(os.environ["HOME"], "TXPipe")
sys.path.append(txpipe_dir)

import matplotlib.pyplot as plt
import h5py
import numpy as np
from txpipe.data_types import HDFFile, ShearCatalog, FiducialCosmology
import sacc
import yaml
from pprint import pprint
import time
import ceci
from txpipe.twopoint import TXTwoPoint

%matplotlib inline

In [ ]:
class TXTwoPointRand(TXTwoPoint):
    name = "TXTwoPointRand"
    inputs = [
        ("random_catalog", HDFFile),
        ("shear_catalog", ShearCatalog),
        ("fiducial_cosmology", FiducialCosmology)
    ]
    
    outputs = [("twopoint_gamma_x", SACCFile)]
    
    config_options = {
          # "binning_scale": "Log"   # idk if i even need this
          "max_sep": 250.0, # Mpc
          "min_sep": 10.0, # Mpc
          "nbins": 15,
          "redshift_bin_edges": [0.4, 0.8, 1.2],
          "richness_bin_edges": [20, 30, 200],
          "units": "arcmin",
          "chunk_rows": 100000,
          "nside": 64
          "pixelization": healpix
          "sparse": True
    }

    def make_random_catalog(self, i):
        # As with the lens catalog version, we add the r_col keyword
        # compare to the parent class
        import treecorr

        if not self.config["use_randoms"]:
            return None

        rancat = treecorr.Catalog(
            self.get_input("binned_random_catalog"),
            ext=f"/randoms/bin_{i}",
            ra_col="ra",
            dec_col="dec",
            ra_units="degree",
            dec_units="degree",
            patch_centers=self.get_input("patch_centers"),
            save_patch_dir=self.get_patch_dir("binned_random_catalog", i),
        )
        return rancat